# LeetCode 1360 — Number of Days Between Two Dates bằng Python

**Link đề:** https://leetcode.com/problems/number-of-days-between-two-dates/
**Độ khó:** Easy · **Chủ đề:** Math, String

```python
class Solution:
    def daysBetweenDates(self, date1: str, date2: str) -> int:
        ...
```

### Mục tiêu học
1. Tách chuỗi ngày `"YYYY-MM-DD"` bằng `split`, `map(int, ...)`, unpacking.
2. Nắm chắc **quy tắc năm nhuận** (chia hết cho 4, trừ thế kỷ không chia hết cho 400).
3. Hiểu kỹ thuật then chốt: **đổi mỗi ngày thành "số thứ tự ngày" tính từ một mốc cố định** rồi lấy hiệu tuyệt đối.
4. Đi qua 4 lời giải: dùng `datetime` → đếm từng ngày (mô phỏng) → cộng theo năm/tháng → công thức đóng `O(1)`.
5. Trace, chứng minh, phân tích Time/Space, và tự giải biến thể (ngày trong tuần, cộng thêm k ngày).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Viết chương trình đếm **số ngày giữa hai ngày**. Hai ngày được cho dạng chuỗi `"YYYY-MM-DD"`.

| date1 | date2 | Output |
|---|---|---|
| `"2019-06-29"` | `"2019-06-30"` | `1` |
| `"2020-01-15"` | `"2019-12-31"` | `15` |

**Constraints:** các ngày đều **hợp lệ** và nằm trong khoảng năm **1971 – 2100**.

Nhận xét:
- Ví dụ 2: `date1` **sau** `date2` mà kết quả vẫn dương → đáp án là **khoảng cách** (giá trị tuyệt đối).
- Hai ngày bằng nhau → `0`.

Câu hỏi đặt ra:
- Đếm ngày khi hai ngày khác tháng, khác năm, qua năm nhuận thế nào?
- Có cách nào biến bài "hiệu hai ngày" thành bài "hiệu hai số"?

## 2. Nền tảng Python: tách chuỗi ngày

- `s.split("-")` cắt chuỗi tại mỗi dấu `-`, trả về **list chuỗi**.
- `int("06")` → `6` (số 0 đầu được bỏ qua).
- `map(int, list)` áp `int` cho từng phần tử.
- `y, m, d = ...` là **unpacking**: gán 3 phần tử vào 3 biến.

In [1]:
s = "2019-06-29"
parts = s.split("-")
print(parts)                    # ['2019', '06', '29']
y, m, d = map(int, parts)
print(y, m, d, type(y))

['2019', '06', '29']
2019 6 29 <class 'int'>


In [2]:
def parse_date(s):
    y, m, d = map(int, s.split("-"))
    return y, m, d

print(parse_date("2020-01-15"))
print(parse_date("1971-01-01"))

(2020, 1, 15)
(1971, 1, 1)


## 3. Nền tảng lịch: năm nhuận và số ngày mỗi tháng

**Năm nhuận** (leap year) có 366 ngày (tháng 2 có 29 ngày). Quy tắc lịch Gregory:
- Chia hết cho 4 → nhuận,
- **trừ** khi chia hết cho 100 → không nhuận,
- **trừ tiếp** khi chia hết cho 400 → vẫn nhuận.

Ví dụ: 2020 nhuận; 1900 **không** nhuận; 2000 nhuận; 2100 **không** nhuận (2100 nằm trong miền đề bài — phải xử lý đúng!).

`%` là phép chia lấy dư: `2020 % 4 == 0`.

In [3]:
def is_leap(y):
    return y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)

for y in [2019, 2020, 1900, 2000, 2100, 1972]:
    print(y, "nhuận" if is_leap(y) else "không nhuận")

2019 không nhuận
2020 nhuận
1900 không nhuận
2000 nhuận
2100 không nhuận
1972 nhuận


Số ngày mỗi tháng (tháng 1 → 12): `31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31` (tháng 2 là 29 nếu nhuận). Ta lưu thành list; vì list đếm từ 0 nên **tháng `m` ở chỉ số `m - 1`**.

In [4]:
MONTH_DAYS = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

def days_in_month(y, m):
    if m == 2 and is_leap(y):
        return 29
    return MONTH_DAYS[m - 1]

print(days_in_month(2019, 2), days_in_month(2020, 2), days_in_month(2100, 2), days_in_month(2019, 6))
print("Tổng ngày năm thường:", sum(MONTH_DAYS))

28 29 28 30
Tổng ngày năm thường: 365


## 4. Cách 1 — Dùng thư viện chuẩn `datetime`

Python có module `datetime` (thư viện chuẩn, không cần cài):
- `date.fromisoformat("2019-06-29")` đọc chuỗi ISO thành đối tượng `date`.
- Trừ hai `date` cho ra `timedelta`, thuộc tính `.days` là số ngày (có thể âm).
- `abs(...)` lấy trị tuyệt đối.

In [5]:
from datetime import date

d1 = date.fromisoformat("2020-01-15")
d2 = date.fromisoformat("2019-12-31")
delta = d1 - d2
print(type(delta).__name__, delta, delta.days)
print((d2 - d1).days, abs((d2 - d1).days))

timedelta 15 days, 0:00:00 15
-15 15


In [6]:
def days_between_datetime(date1, date2):
    return abs((date.fromisoformat(date1) - date.fromisoformat(date2)).days)

print(days_between_datetime("2019-06-29", "2019-06-30"))   # 1
print(days_between_datetime("2020-01-15", "2019-12-31"))   # 15

1
15


Cách này đúng và được LeetCode chấp nhận, nhưng **giấu hết logic**. Trong phỏng vấn thường được yêu cầu tự tính. Ta sẽ dùng nó làm **"oracle"** (đáp án chuẩn) để kiểm tra các cách tự viết.

`date.toordinal()` trả **số thứ tự ngày** tính từ `0001-01-01` (ngày 1). Đây chính là ý tưởng ta sẽ tự xây: đổi ngày thành một con số.

In [7]:
print(date(1, 1, 1).toordinal(), date(1971, 1, 1).toordinal(), date(2019, 6, 29).toordinal())

1 719528 737239


## 5. Cách 2 — Baseline: mô phỏng "lật lịch" từng ngày

Ý tưởng trực giác: bắt đầu từ ngày **nhỏ hơn**, cứ tăng 1 ngày cho tới khi bằng ngày lớn hơn, đếm số lần tăng.

Tăng 1 ngày `(y, m, d)`:
- `d += 1`; nếu `d` vượt số ngày của tháng → `d = 1`, `m += 1`;
- nếu `m > 12` → `m = 1`, `y += 1`.

So sánh hai ngày dạng tuple `(y, m, d)`: Python so tuple **từ trái sang phải** nên `(2019,12,31) < (2020,1,15)` đúng như thứ tự thời gian.

In [8]:
def next_day(y, m, d):
    d += 1
    if d > days_in_month(y, m):
        d = 1
        m += 1
        if m > 12:
            m = 1
            y += 1
    return y, m, d

print(next_day(2019, 6, 29), next_day(2019, 12, 31), next_day(2020, 2, 28), next_day(2100, 2, 28))
print((2019, 12, 31) < (2020, 1, 15))

(2019, 6, 30) (2020, 1, 1) (2020, 2, 29) (2100, 3, 1)
True


In [9]:
def days_between_simulate(date1, date2):
    a, b2 = parse_date(date1), parse_date(date2)
    if a > b2:
        a, b2 = b2, a          # đổi chỗ để a là ngày sớm hơn
    count = 0
    while a != b2:
        a = next_day(*a)       # *a: "rải" tuple thành 3 tham số
        count += 1
    return count

print(days_between_simulate("2019-06-29", "2019-06-30"))   # 1
print(days_between_simulate("2020-01-15", "2019-12-31"))   # 15

1
15


Hai cú pháp mới:
- `a, b2 = b2, a` — **hoán đổi** hai biến trong một dòng.
- `next_day(*a)` — dấu `*` **rải** tuple `(y, m, d)` thành 3 đối số `next_day(y, m, d)`.

**Nút thắt:** khoảng cách tối đa (1971 → 2100) khoảng 47.000 ngày → vòng lặp ~47.000 bước. Vẫn chạy được, nhưng mỗi lần hỏi phải "lật lịch" lại từ đầu. Ta muốn tính **trực tiếp**.

## 6. Bước đột phá — đổi ngày thành con số

**Insight:** chọn một **mốc cố định** (ví dụ `1971-01-01`). Định nghĩa `f(date)` = số ngày từ mốc tới `date`. Khi đó:

```text
khoảng cách(date1, date2) = |f(date1) - f(date2)|
```

Giống như đo khoảng cách giữa hai điểm trên trục số: đo vị trí mỗi điểm so với gốc `0`, rồi trừ cho nhau. Bài toán "hai ngày" trở thành bài "hai số".

Ví dụ nhỏ: nếu `f("2019-12-31") = X` thì `f("2020-01-15") = X + 15` → hiệu là 15. Mốc chọn gì cũng được, miễn **cùng một mốc** cho cả hai ngày.

## 7. Cách 3 (chính) — Tính `f(date)` bằng cộng theo năm và tháng

`f(y, m, d)` = (số ngày của các **năm trọn vẹn** từ 1971 tới `y - 1`)
 + (số ngày của các **tháng trọn vẹn** từ tháng 1 tới `m - 1` của năm `y`)
 + `d`.

Mổ xẻ từng đoạn:

In [10]:
def days_of_full_years(y):
    total = 0
    for year in range(1971, y):              # 1971 .. y-1
        total += 366 if is_leap(year) else 365
    return total

def days_of_full_months(y, m):
    total = 0
    for month in range(1, m):                # 1 .. m-1
        total += days_in_month(y, month)
    return total

print("Năm trọn trước 1973:", days_of_full_years(1973), "(1971: 365, 1972 nhuận: 366)")
print("Tháng trọn trước 03/2020:", days_of_full_months(2020, 3), "(31 + 29)")

Năm trọn trước 1973: 731 (1971: 365, 1972 nhuận: 366)
Tháng trọn trước 03/2020: 60 (31 + 29)


In [11]:
def day_number(s):
    y, m, d = parse_date(s)
    return days_of_full_years(y) + days_of_full_months(y, m) + d

def days_between_count(date1, date2):
    return abs(day_number(date1) - day_number(date2))

print(day_number("1971-01-01"))                            # 1
print(days_between_count("2019-06-29", "2019-06-30"))      # 1
print(days_between_count("2020-01-15", "2019-12-31"))      # 15

1
1
15


Mốc `1971-01-01` có `f = 1` (vì cộng `d = 1`). Không sao: hằng số cộng thêm triệt tiêu khi lấy hiệu.

`range(1971, y)` đi từ 1971 tới `y - 1` (không gồm `y`) — đúng "các năm **trọn vẹn** trước năm `y`". Tương tự `range(1, m)` là các tháng trọn trước tháng `m`.

## 8. TRACE từng bước — `f("2020-01-15")` và `f("2019-12-31")`

In [12]:
def trace_day_number(s):
    y, m, d = parse_date(s)
    print(f"Ngày {s}: y={y}, m={m}, d={d}")
    years = days_of_full_years(y)
    leaps = [yr for yr in range(1971, y) if is_leap(yr)]
    print(f"  Năm trọn 1971..{y-1}: {y - 1971} năm, trong đó {len(leaps)} năm nhuận → {years} ngày")
    months = 0
    for month in range(1, m):
        months += days_in_month(y, month)
        print(f"  + tháng {month}: {days_in_month(y, month)} ngày (cộng dồn {months})")
    if m == 1:
        print("  Không có tháng trọn nào trước tháng 1")
    total = years + months + d
    print(f"  f = {years} + {months} + {d} = {total}")
    return total

a = trace_day_number("2020-01-15")
print()
b2 = trace_day_number("2019-12-31")
print("\n|f1 - f2| =", abs(a - b2))

Ngày 2020-01-15: y=2020, m=1, d=15
  Năm trọn 1971..2019: 49 năm, trong đó 12 năm nhuận → 17897 ngày
  Không có tháng trọn nào trước tháng 1
  f = 17897 + 0 + 15 = 17912

Ngày 2019-12-31: y=2019, m=12, d=31
  Năm trọn 1971..2018: 48 năm, trong đó 12 năm nhuận → 17532 ngày
  + tháng 1: 31 ngày (cộng dồn 31)
  + tháng 2: 28 ngày (cộng dồn 59)
  + tháng 3: 31 ngày (cộng dồn 90)
  + tháng 4: 30 ngày (cộng dồn 120)
  + tháng 5: 31 ngày (cộng dồn 151)
  + tháng 6: 30 ngày (cộng dồn 181)
  + tháng 7: 31 ngày (cộng dồn 212)
  + tháng 8: 31 ngày (cộng dồn 243)
  + tháng 9: 30 ngày (cộng dồn 273)
  + tháng 10: 31 ngày (cộng dồn 304)
  + tháng 11: 30 ngày (cộng dồn 334)
  f = 17532 + 334 + 31 = 17897

|f1 - f2| = 15


Bảng tóm tắt:

| Ngày | Năm trọn | Tháng trọn | d | f |
|---|---|---|---|---|
| 2020-01-15 | 1971..2019 (49 năm, 12 nhuận) = 17897 | 0 | 15 | 17912 |
| 2019-12-31 | 1971..2018 (48 năm, 12 nhuận) = 17532 | 334 | 31 | 17897 |

`17912 - 17897 = 15` ✓. Lưu ý `2019-12-31` có `f` đúng bằng số ngày của 49 năm trọn — hợp lý vì đó là **ngày cuối cùng** của năm 2019.

## 9. Cách 4 — Công thức đóng `O(1)` (không vòng lặp)

### 9.1. Đếm năm nhuận bằng công thức

Số năm nhuận trong `1..Y` = `Y//4 - Y//100 + Y//400` (bao hàm – loại trừ: chia hết 4, bỏ chia hết 100, thêm lại chia hết 400). Vậy số năm nhuận trong `[1971, y-1]` = `L(y-1) - L(1970)`.

### 9.2. Tháng trọn bằng **mảng cộng dồn** (prefix sum)

`CUM[m]` = tổng ngày của các tháng `1..m` năm thường: `[0, 31, 59, 90, ...]`. Tháng trọn trước tháng `m` = `CUM[m-1]`, cộng thêm 1 nếu năm nhuận và `m > 2`.

In [13]:
def leaps_upto(Y):
    return Y // 4 - Y // 100 + Y // 400

print("Năm nhuận 1971..2019 (công thức):", leaps_upto(2019) - leaps_upto(1970))
print("Năm nhuận 1971..2019 (đếm):     ", sum(is_leap(y) for y in range(1971, 2020)))

CUM = [0]
for x in MONTH_DAYS:
    CUM.append(CUM[-1] + x)
print("CUM =", CUM)

Năm nhuận 1971..2019 (công thức): 12
Năm nhuận 1971..2019 (đếm):      12
CUM = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334, 365]


In [14]:
def day_number_formula(s):
    y, m, d = map(int, s.split("-"))
    years = (y - 1971) * 365 + leaps_upto(y - 1) - leaps_upto(1970)
    months = CUM[m - 1] + (1 if m > 2 and is_leap(y) else 0)
    return years + months + d

def days_between_formula(date1, date2):
    return abs(day_number_formula(date1) - day_number_formula(date2))

print(day_number_formula("2020-01-15"), day_number_formula("2019-12-31"))
print(days_between_formula("2019-06-29", "2019-06-30"), days_between_formula("2020-01-15", "2019-12-31"))

17912 17897
1 15


### 9.3. (Mở rộng) Công thức "days from civil" — dịch tháng 3 lên đầu năm

Một mẹo cổ điển: coi **tháng 3 là tháng đầu năm**, tháng 1–2 thuộc "năm trước". Khi đó ngày nhuận (29/2) luôn ở **cuối** năm và số ngày của các tháng 3→2 có thể tính bằng công thức `(153 * mp + 2) // 5`. Không cần bảng tháng.

Ta chỉ minh hoạ để mở rộng hiểu biết; bạn không cần thuộc.

In [15]:
def days_from_civil(y, m, d):
    y -= m <= 2                      # True = 1: tháng 1, 2 thuộc năm trước
    mp = (m + 9) % 12                # tháng 3 → 0, ..., tháng 2 → 11
    doy = (153 * mp + 2) // 5 + d - 1
    return 365 * y + y // 4 - y // 100 + y // 400 + doy

for s in ["2019-12-31", "2020-01-15", "2020-03-01"]:
    y, m, d = parse_date(s)
    print(s, days_from_civil(y, m, d), "| ordinal - civil =", date(y, m, d).toordinal() - days_from_civil(y, m, d))

2019-12-31 737729 | ordinal - civil = -305
2020-01-15 737744 | ordinal - civil = -305
2020-03-01 737790 | ordinal - civil = -305


Hiệu `ordinal - civil` là **hằng số** cho mọi ngày → hai cách đánh số chỉ lệch nhau một mốc, nên hiệu giữa hai ngày luôn giống nhau.

## 10. Code nộp LeetCode

Bản tự tính (Cách 4, `O(1)`):

In [16]:
class Solution:
    def daysBetweenDates(self, date1: str, date2: str) -> int:
        cum = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]

        def is_leap(y: int) -> bool:
            return y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)

        def leaps(y: int) -> int:
            return y // 4 - y // 100 + y // 400

        def f(s: str) -> int:
            y, m, d = map(int, s.split("-"))
            days = (y - 1971) * 365 + leaps(y - 1) - leaps(1970)
            days += cum[m - 1] + (1 if m > 2 and is_leap(y) else 0)
            return days + d

        return abs(f(date1) - f(date2))

sol = Solution()
print(sol.daysBetweenDates("2019-06-29", "2019-06-30"))   # 1
print(sol.daysBetweenDates("2020-01-15", "2019-12-31"))   # 15

1
15


Bản dùng thư viện (ngắn nhất, cũng Accepted):

In [17]:
from datetime import date

class SolutionDatetime:
    def daysBetweenDates(self, date1: str, date2: str) -> int:
        return abs((date.fromisoformat(date1) - date.fromisoformat(date2)).days)

print(SolutionDatetime().daysBetweenDates("2020-01-15", "2019-12-31"))

15


## 11. Vì sao đúng?

**Mệnh đề:** với hai ngày liên tiếp `D` và `next(D)`, ta có `f(next(D)) = f(D) + 1`.

- Cùng tháng: `d` tăng 1, các phần khác giữ nguyên → `+1`.
- Sang tháng mới (`d` = ngày cuối tháng `m` → `1` tháng `m+1`): phần tháng trọn tăng đúng `days_in_month(y, m)` = `d` cũ, còn `d` giảm từ `days_in_month` về `1` → tổng `+1`.
- Sang năm mới (31/12 → 1/1): phần năm trọn tăng số ngày của năm `y` (= 334 + 31 hoặc 335 + 31), phần tháng trọn từ 334/335 về 0, `d` từ 31 về 1 → tổng `+1`.

Vậy `f` **tăng đúng 1 mỗi ngày**, tức `f(D2) - f(D1)` bằng số lần "lật lịch" từ `D1` tới `D2` = số ngày giữa chúng. Lấy `abs` để không phụ thuộc thứ tự.

Công thức đếm năm nhuận đúng theo nguyên lý **bao hàm – loại trừ**: các năm chia hết 400 bị trừ ở bước "chia hết 100" nên phải cộng lại.

### 11.1. Phân tích Time / Space

Gọi `Y` = số năm giữa mốc và ngày (≤ 130), `D` = số ngày giữa hai ngày (≤ ~47.500).

| Cách | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|
| 1. `datetime` | `O(1)` | `O(1)` | phụ thuộc thư viện |
| 2. Mô phỏng từng ngày | `O(D)` | `O(1)` | chậm nhất, dễ hiểu nhất |
| 3. Cộng theo năm/tháng | `O(Y + 12)` | `O(1)` | baseline tốt |
| 4. Công thức đóng | `O(1)` | `O(1)` | tối ưu |

Do miền năm bị giới hạn (1971–2100), mọi cách đều rất nhanh trên LeetCode; khác biệt chủ yếu là **tư duy**.

In [18]:
print("Khoảng cách lớn nhất trong miền đề:", days_between_formula("1971-01-01", "2100-12-31"), "ngày")

Khoảng cách lớn nhất trong miền đề: 47481 ngày


## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.daysBetweenDates("2019-06-29", "2019-06-30"), 1)
check("đề 2 (date1 > date2)", sol.daysBetweenDates("2020-01-15", "2019-12-31"), 15)
check("cùng ngày", sol.daysBetweenDates("2000-05-05", "2000-05-05"), 0)
check("qua 29/2 năm nhuận", sol.daysBetweenDates("2020-02-28", "2020-03-01"), 2)
check("qua 28/2 năm thường", sol.daysBetweenDates("2019-02-28", "2019-03-01"), 1)
check("2100 KHÔNG nhuận", sol.daysBetweenDates("2100-02-28", "2100-03-01"), 1)
check("2000 nhuận", sol.daysBetweenDates("2000-02-28", "2000-03-01"), 2)
check("trọn năm nhuận", sol.daysBetweenDates("2020-01-01", "2021-01-01"), 366)
check("biên miền đề", sol.daysBetweenDates("1971-01-01", "2100-12-31"), 47481)

PASS đề 1: 1
PASS đề 2 (date1 > date2): 15
PASS cùng ngày: 0
PASS qua 29/2 năm nhuận: 2
PASS qua 28/2 năm thường: 1
PASS 2100 KHÔNG nhuận: 1
PASS 2000 nhuận: 2
PASS trọn năm nhuận: 366
PASS biên miền đề: 47481


In [20]:
import random

methods = [days_between_simulate, days_between_count, days_between_formula,
           Solution().daysBetweenDates, SolutionDatetime().daysBetweenDates]
lo, hi = date(1971, 1, 1).toordinal(), date(2100, 12, 31).toordinal()
rng = random.Random(1360)
for _ in range(300):
    d1 = date.fromordinal(rng.randint(lo, hi)).isoformat()
    d2 = date.fromordinal(rng.randint(lo, hi)).isoformat()
    expected = days_between_datetime(d1, d2)
    for f in methods[1:]:
        assert f(d1, d2) == expected, (f, d1, d2)
# mô phỏng chậm hơn → chỉ kiểm tra cặp gần nhau
for _ in range(100):
    o = rng.randint(lo, hi - 800)
    d1 = date.fromordinal(o).isoformat()
    d2 = date.fromordinal(o + rng.randint(0, 800)).isoformat()
    assert days_between_simulate(d1, d2) == days_between_datetime(d1, d2)
print("Đối chiếu ngẫu nhiên với datetime: PASS")

Đối chiếu ngẫu nhiên với datetime: PASS


`date.fromordinal(k)` là phép ngược của `toordinal()`, `isoformat()` trả lại chuỗi `"YYYY-MM-DD"` — tiện để sinh ngày ngẫu nhiên **hợp lệ**.

### Kiểm tra mệnh đề "f tăng đúng 1 mỗi ngày" trên toàn miền đề

In [21]:
prev = None
cur = (1971, 1, 1)
n_days = 0
while cur <= (2100, 12, 31):
    s = f"{cur[0]:04d}-{cur[1]:02d}-{cur[2]:02d}"
    v = day_number_formula(s)
    if prev is not None:
        assert v == prev + 1, s
    prev = v
    cur = next_day(*cur)
    n_days += 1
print("Đã kiểm tra", n_days, "ngày liên tiếp: f tăng đúng 1 mỗi ngày")

Đã kiểm tra 47482 ngày liên tiếp: f tăng đúng 1 mỗi ngày


`f"{x:02d}"` in số nguyên rộng 2 ký tự, đệm `0` bên trái: `6 → "06"`.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Quy tắc năm nhuận thiếu điều kiện
```python
def is_leap(y): return y % 4 == 0      # SAI với 1900, 2100
```
2100 nằm trong miền đề → bài sai ở test có tháng 2/2100.

In [22]:
def wrong_is_leap(y):
    return y % 4 == 0

print("2100 — sai:", wrong_is_leap(2100), "| đúng:", is_leap(2100))

2100 — sai: True | đúng: False


### Lỗi 2: Quên `abs`
Ví dụ 2 có `date1 > date2` → kết quả âm `-15` nếu chỉ trừ.

### Lỗi 3: Lệch một tháng khi cộng tháng trọn
```python
for month in range(1, m + 1):   # SAI: cộng cả tháng m chưa trọn
```

In [23]:
def wrong_day_number(s):
    y, m, d = parse_date(s)
    months = sum(days_in_month(y, k) for k in range(1, m + 1))   # thừa tháng m
    return days_of_full_years(y) + months + d

print("Sai:", abs(wrong_day_number("2019-06-30") - wrong_day_number("2019-07-01")),
      "| Đúng:", days_between_count("2019-06-30", "2019-07-01"))

Sai: 2 | Đúng: 1


Hai ngày cùng tháng thì lỗi này **không lộ** (cùng bị cộng thừa như nhau) — đó là lý do phải test cặp ngày **khác tháng**.

### Lỗi 4: Cộng ngày nhuận cho tháng 1–2
Ngày 29/2 chỉ ảnh hưởng các ngày **từ tháng 3 trở đi**. Viết `CUM[m-1] + is_leap(y)` (không kiểm `m > 2`) sẽ lệch 1 cho tháng 1–2 năm nhuận.

In [24]:
def wrong_formula(s):
    y, m, d = map(int, s.split("-"))
    years = (y - 1971) * 365 + leaps_upto(y - 1) - leaps_upto(1970)
    return years + CUM[m - 1] + is_leap(y) + d      # thiếu điều kiện m > 2

print("Sai:", abs(wrong_formula("2019-12-31") - wrong_formula("2020-01-01")),
      "| Đúng:", days_between_formula("2019-12-31", "2020-01-01"))

Sai: 2 | Đúng: 1


### Lỗi 5: So sánh ngày bằng chuỗi khi định dạng không đồng nhất
Chuỗi `"YYYY-MM-DD"` **có đệm 0** so sánh chuỗi được (`"2019-12-31" < "2020-01-15"`), nhưng nếu dữ liệu là `"2019-9-5"` thì `"2019-9-5" > "2019-10-1"` (vì ký tự `'9' > '1'`). Luôn đổi sang số trước khi so sánh.

In [25]:
print("2019-9-5" > "2019-10-1", "| theo số:", (2019, 9, 5) > (2019, 10, 1))

True | theo số: False


### Mẹo nhận dạng pattern

- **"Khoảng cách giữa hai mốc thời gian/vị trí"** → đổi mỗi mốc thành **toạ độ trên trục số** so với một gốc chung rồi trừ.
- Cộng dồn nhiều đoạn có độ dài khác nhau (tháng) → **prefix sum** (`CUM`).
- "Đếm số nguyên trong `[1, Y]` chia hết cho a, trừ b, cộng c" → **bao hàm – loại trừ** với `Y // a`.
- Bài anh em: *Day of the Year* (1154), *Day of the Week* (1185), *Count Days Spent Together* (2409).

## 14. Biến thể

### 14.1. Ngày thứ mấy trong năm (LeetCode 1154)

In [26]:
def day_of_year(s):
    y, m, d = parse_date(s)
    return CUM[m - 1] + (1 if m > 2 and is_leap(y) else 0) + d

print(day_of_year("2019-01-09"), day_of_year("2019-02-10"), day_of_year("2020-12-31"))   # 9 41 366

9 41 366


### 14.2. Thứ trong tuần (LeetCode 1185)
Biết `1971-01-01` là **thứ Sáu**. Dùng `f` của ta: thứ = `(index_thứ_sáu + f(date) - 1) % 7`.

In [27]:
WEEK = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]

def day_of_week(s):
    offset = day_number_formula(s) - 1          # 0 cho 1971-01-01
    return WEEK[(5 + offset) % 7]               # 5 = Friday

print(day_of_week("1971-01-01"), day_of_week("2019-08-31"), day_of_week("2026-11-18"))
print("Kiểm tra bằng datetime:", date(2026, 11, 18).strftime("%A"))

Friday Saturday Wednesday
Kiểm tra bằng datetime: Wednesday


### 14.3. Cộng `k` ngày vào một ngày
Phép ngược của `f`: từ số ngày tìm lại `(y, m, d)` — trừ dần số ngày của từng năm, rồi từng tháng.

In [28]:
def from_day_number(n):
    y = 1971
    while n > (366 if is_leap(y) else 365):
        n -= 366 if is_leap(y) else 365
        y += 1
    m = 1
    while n > days_in_month(y, m):
        n -= days_in_month(y, m)
        m += 1
    return f"{y:04d}-{m:02d}-{n:02d}"

def add_days(s, k):
    return from_day_number(day_number_formula(s) + k)

print(add_days("2019-12-31", 15), add_days("2020-02-28", 1), add_days("2100-02-28", 1))

2020-01-15 2020-02-29 2100-03-01


## 15. Bài tập (TODO — đáp án ở cuối)

### Bài 1 — Tính tay
Tính số ngày giữa `"2021-02-27"` và `"2021-03-02"`. Rồi với `"2024-02-27"` và `"2024-03-02"`. Vì sao khác?

### Bài 2 — Năm nhuận
Viết `exercise_is_leap(y)` và tự kiểm với 1900, 2000, 2020, 2100.

In [29]:
def exercise_is_leap(y):
    # TODO: chia hết 4, trừ chia hết 100, cộng lại chia hết 400
    pass

# Gợi ý test:
# assert [exercise_is_leap(y) for y in [1900, 2000, 2020, 2021, 2100]] == [False, True, True, False, False]

### Bài 3 — Hoàn thiện `f(date)` bằng vòng lặp

In [30]:
def exercise_day_number(s):
    # TODO 1: tách y, m, d
    # TODO 2: cộng ngày các năm trọn 1971..y-1
    # TODO 3: cộng ngày các tháng trọn 1..m-1 (nhớ tháng 2 năm nhuận)
    # TODO 4: cộng d
    pass

# Gợi ý test:
# assert abs(exercise_day_number("2020-01-15") - exercise_day_number("2019-12-31")) == 15

### Bài 4 — Đếm số ngày 29/2 trong khoảng
Đếm số ngày `29/02` nằm trong đoạn `[date1, date2]` (gồm hai đầu, `date1 <= date2`).

In [31]:
def exercise_count_feb29(date1, date2):
    # TODO
    pass

# Gợi ý test:
# assert exercise_count_feb29("2019-01-01", "2021-12-31") == 1
# assert exercise_count_feb29("2020-03-01", "2024-02-29") == 1
# assert exercise_count_feb29("2096-01-01", "2100-12-31") == 1

### Bài 5 — Số ngày hai khoảng giao nhau (ý tưởng LeetCode 2409)
Cho hai khoảng ngày `[a1, a2]` và `[b1, b2]` (gồm hai đầu), trả về số ngày chung.

In [32]:
def exercise_overlap_days(a1, a2, b1, b2):
    # TODO: max(đầu), min(cuối), đổi sang số
    pass

# Gợi ý test:
# assert exercise_overlap_days("2020-01-01", "2020-01-10", "2020-01-05", "2020-01-20") == 6
# assert exercise_overlap_days("2020-01-01", "2020-01-03", "2020-01-04", "2020-01-05") == 0

### Bài 6 — Mini challenge: thứ của ngày bằng `f` tự viết
Viết `exercise_day_of_week(s)` không dùng `datetime`, biết `1971-01-01` là thứ Sáu.

In [33]:
def exercise_day_of_week(s):
    # TODO
    pass

# Gợi ý test:
# assert exercise_day_of_week("2019-08-31") == "Saturday"
# assert exercise_day_of_week("1993-08-15") == "Sunday"

## 16. Gợi ý 3 mức

**Bài 1** — (1) Liệt kê ngày. (2) 2021 không nhuận, 2024 nhuận. (3) 3 và 4.

**Bài 2** — (1) Ba điều kiện lồng nhau. (2) `y % 4 == 0 and (...)`. (3) `y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)`.

**Bài 3** — (1) Ba phần: năm, tháng, ngày. (2) `range(1971, y)` và `range(1, m)`. (3) dùng `366 if is_leap(year) else 365` và `days_in_month`.

**Bài 4** — (1) Duyệt từng năm trong `[y1, y2]`. (2) Năm nhuận nào có ngày `y-02-29` nằm trong đoạn thì đếm. (3) So sánh tuple `(y1,m1,d1) <= (y,2,29) <= (y2,m2,d2)`.

**Bài 5** — (1) Giao của hai đoạn trên trục số. (2) `start = max(f(a1), f(b1))`, `end = min(f(a2), f(b2))`. (3) `max(0, end - start + 1)`.

**Bài 6** — (1) `f(1971-01-01)` ứng với thứ Sáu. (2) Mỗi ngày lệch thêm 1 thứ, chu kỳ 7. (3) `WEEK[(5 + f(s) - 1) % 7]`.

## 17. Đáp án tham khảo

**Bài 1:** 2021: `27/2 → 28/2 → 1/3 → 2/3` = **3** ngày. 2024 nhuận có thêm 29/2 → **4** ngày.

In [34]:
assert days_between_formula("2021-02-27", "2021-03-02") == 3
assert days_between_formula("2024-02-27", "2024-03-02") == 4

def answer_is_leap(y):
    return y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)

def answer_day_number(s):
    y, m, d = map(int, s.split("-"))
    total = 0
    for year in range(1971, y):
        total += 366 if answer_is_leap(year) else 365
    for month in range(1, m):
        total += 29 if (month == 2 and answer_is_leap(y)) else MONTH_DAYS[month - 1]
    return total + d

def answer_count_feb29(date1, date2):
    a, b_ = parse_date(date1), parse_date(date2)
    return sum(1 for y in range(a[0], b_[0] + 1)
               if answer_is_leap(y) and a <= (y, 2, 29) <= b_)

def answer_overlap_days(a1, a2, b1, b2):
    start = max(answer_day_number(a1), answer_day_number(b1))
    end = min(answer_day_number(a2), answer_day_number(b2))
    return max(0, end - start + 1)

def answer_day_of_week(s):
    return WEEK[(5 + answer_day_number(s) - 1) % 7]

assert [answer_is_leap(y) for y in [1900, 2000, 2020, 2021, 2100]] == [False, True, True, False, False]
assert abs(answer_day_number("2020-01-15") - answer_day_number("2019-12-31")) == 15
assert answer_count_feb29("2019-01-01", "2021-12-31") == 1
assert answer_count_feb29("2020-03-01", "2024-02-29") == 1
assert answer_count_feb29("2096-01-01", "2100-12-31") == 1
assert answer_overlap_days("2020-01-01", "2020-01-10", "2020-01-05", "2020-01-20") == 6
assert answer_overlap_days("2020-01-01", "2020-01-03", "2020-01-04", "2020-01-05") == 0
assert answer_day_of_week("2019-08-31") == "Saturday"
assert answer_day_of_week("1993-08-15") == "Sunday"

rng = random.Random(5)
for _ in range(200):
    s = date.fromordinal(rng.randint(lo, hi))
    assert answer_day_of_week(s.isoformat()) == s.strftime("%A")
print("Đáp án bài tập: PASS toàn bộ")

Đáp án bài tập: PASS toàn bộ


## 18. Tổng kết — template nhớ nhanh + checklist

```python
def is_leap(y): return y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)
CUM = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
def f(s):                                     # số ngày tính từ mốc 1971-01-01
    y, m, d = map(int, s.split("-"))
    L = lambda Y: Y // 4 - Y // 100 + Y // 400
    return (y - 1971) * 365 + L(y - 1) - L(1970) + CUM[m - 1] + (m > 2 and is_leap(y)) + d
answer = abs(f(date1) - f(date2))
```

**Checklist trước khi submit**
- [ ] Năm nhuận đủ 3 điều kiện (2100 không nhuận, 2000 nhuận).
- [ ] Cộng ngày nhuận chỉ khi `m > 2`.
- [ ] Tháng trọn là `1..m-1`, năm trọn là `1971..y-1`.
- [ ] Có `abs(...)` vì `date1` có thể sau `date2`.
- [ ] Đã test cùng ngày, qua 29/2, qua giao thừa, hai đầu miền đề.